In [94]:
import json
import time

import numpy as np
import pandas as pd
import requests
import redis
from redis.commands.search.field import (
    NumericField,
    TagField,
    TextField,
    VectorField,
)
from redis.commands.search.indexDefinition import IndexDefinition, IndexType
from redis.commands.search.query import Query
# from sentence_transformers import SentenceTransformer
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter


In [95]:
# Connect to Redis
client = redis.Redis(host='127.0.0.1', port=6379, socket_timeout=10)

In [96]:
client.ping()

True

In [97]:
document_after_reading = SimpleDirectoryReader('./example/').load_data()

print("Document ID:", document_after_reading[0].doc_id)
print("Documents : ", document_after_reading[0])

Document ID: cca706a6-bff2-4d07-a2c6-16f6ccf0d2e2
Documents :  Doc ID: cca706a6-bff2-4d07-a2c6-16f6ccf0d2e2
Text: Confidential  9.1 Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT –
ConEnt Audit Name  Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt
Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT BRD Reference No  3.1.1 .1
Objective  To identify discrepancy of count and sum of duration
between Roaming VoiceSMS_MSCvsTAPOUT Sources  • ericsson_msc_roaming •
tapout Frequency of...


In [98]:
base_splitter = SentenceSplitter(chunk_size=400, chunk_overlap=40)

nodes2 = base_splitter.get_nodes_from_documents(document_after_reading)
print(nodes2[0].text)
len(nodes2)


Confidential  9.1 Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt
Audit Name  Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt

Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT
BRD Reference No  3.1.1 .1
Objective  To identify discrepancy of count and sum of duration between Roaming
VoiceSMS_MSCvsTAPOUT
Sources  • ericsson_msc_roaming
• tapout
Frequency of Audit  Daily
Measure Name  DM1 – First Level Recon Between MSC and TAPOUT(Inroamers)
Description  This measure gives one to one mapping of records from both the sources
DM Source -1  ericsson_msc_roaming
Filter Conditions  is_maxis_subscriber = N
and call_duration <> 0
and b_number <> 994
and call_transaction_type<> transit -mt
and imsi notlike 50218%
and imsi notlike 502153%
DM Source -2 tapout
Filter Conditions  recordtype_cdr <> GPRS
and duplicate_fl = N
Match Keys  ericsson_msc_roaming  tapout
imsi
start_dttm
call_duration
recipient_plmn_id  Imsi
call_event_start_time
chargeable_units
recipient
Column Selection  ericsson_msc_roaming


450

In [99]:
# Get embeddings for all the nodes
contents = []
for i in range(0,len(nodes2)):
    # print(i)
# Define the URL and the data
    url = 'http://localhost:11434/api/embeddings'
    data = {
      "model": "jmorgan/gte-small:latest", 
      "prompt": nodes2[i].get_content()
    }
    
    # Send the POST request
    response = requests.post(url, json=data)
    
    
    # Convert response to JSON (dict)
    response_data = response.json()
    response_map = {}
    response_map['text'] = nodes2[i].get_content()
    response_map['embed_text'] = response_data['embedding']
    contents.append(response_map)

# Print or manipulate the data
print(len(contents))
VECTOR_DIMENSION = len(contents[0]['embed_text'])
print(VECTOR_DIMENSION)

450
384


In [101]:
contents[57]

{'text': 'Confidential  file_id_serv\narch_flag\nunrounded_amount\ncell_id_origin\norig_type_id_usg\nnum_records\namount_reduction\nduration\nraw_units_rounded\nadd_implied_decimal\nrerated_dt\ncall_type\nhour_of_day\nexchange_id\nmvpn_call_type\nmvpn_call_type_comp\next_tracking_id\nprimary_units_long\nrerated_amount\na_elt_name\nb_elt_name\na_elt_parent_name\nb_elt_parent_name\ntariff_name\nband_name\ncorridor_plan_id\nsms_partner_network\nelement_id\ncomponent_id\njurisdiction\nis_jurisdiction_in_scope\nMeasure name  QM5 – Rate Mismatched CDRs -RatePlans TM2848 & 78\nDescription  This measure gives Summary of Rate Mismatched CDRs -RatePlans TM2848 & 78\nControl points  QM3 – Rate Mismatched CDRs -RatePlans in Scope\nFilter  (tariff_name like %TalkMore28%\nor tariff_name like %TalkMore48%\nor tariff_name like %TalkMore78%)\nand convert(bigint,rated_units) = raw_units_rounded\nReturn distinct rows\nonly  No\nOutput  record_number\nrecord_address\nrecord_length\nrecord_type\nduplicate_

In [102]:
with open('text_embeddings.json','w') as f :
    json.dump(contents,f)

In [103]:
# Connect to Redis
client = redis.Redis(host='127.0.0.1', port=6379, socket_timeout=10)

In [104]:
client.ping()

True

In [119]:
pipeline = client.pipeline()
for i, items in enumerate(contents, start=1):
    redis_key = f"automobiles:{i:03}"
    pipeline.json().set(redis_key, "$", items)
res = pipeline.execute()
res

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,

In [120]:
res = client.json().get("automobiles:132", "$.text")
res

['round(convert  (float,  (ISNULL  ((total_volume/\n(10*1024))  * pri.price_point,0.0)/100)),  -4)))\nMeasure name  QM11: Q_Duration_match_Volume_Mismatch_Summary\nDescription  This measure gives Duration_match_Volume_Mismatch_Summary\nControl points  QM6: Get Duration matched but data volume not matched\nFilter  NA\nReturn distinct rows\nonly  NO\nOutput  summary_type  = SGSN TAPOUT Duration Match Volume Difference\nstart_dttm  = convert(datetime,convert(date,min_record_opening_time),112)\ntotal_count  = sum (ISNULL (count_imsi_sgsn,0))\ntotal_duration = sum (abs (ISNULL (sum_duration,0) -ISNULL\n(sum_total_call_duration,0)))\ntotal_volume  = sum (abs (ISNULL (sum_data_volume_total,0) -ISNULL\n(sum_chargeable_units,0)))\nRT RT_SGSN_TAPOUT_Summary\n\nr_p3_sgsn_tapout\nMeasure name  QM12: Rev_Get Duration matched but data volume not matched\nDescription  This measure gives Rev_Get  Duration matched but data volume not matched\nControl points  QM11: Q_Duration_match_Volume_Mismatch_Summa

In [121]:
keys = sorted(client.keys("automobiles:*"))

In [122]:
keys

[b'automobiles:001',
 b'automobiles:002',
 b'automobiles:003',
 b'automobiles:004',
 b'automobiles:005',
 b'automobiles:006',
 b'automobiles:007',
 b'automobiles:008',
 b'automobiles:009',
 b'automobiles:010',
 b'automobiles:011',
 b'automobiles:012',
 b'automobiles:013',
 b'automobiles:014',
 b'automobiles:015',
 b'automobiles:016',
 b'automobiles:017',
 b'automobiles:018',
 b'automobiles:019',
 b'automobiles:020',
 b'automobiles:021',
 b'automobiles:022',
 b'automobiles:023',
 b'automobiles:024',
 b'automobiles:025',
 b'automobiles:026',
 b'automobiles:027',
 b'automobiles:028',
 b'automobiles:029',
 b'automobiles:030',
 b'automobiles:031',
 b'automobiles:032',
 b'automobiles:033',
 b'automobiles:034',
 b'automobiles:035',
 b'automobiles:036',
 b'automobiles:037',
 b'automobiles:038',
 b'automobiles:039',
 b'automobiles:040',
 b'automobiles:041',
 b'automobiles:042',
 b'automobiles:043',
 b'automobiles:044',
 b'automobiles:045',
 b'automobiles:046',
 b'automobiles:047',
 b'automobile

In [123]:
schema = (
    TextField("$.text", as_name="text"),
    VectorField(
        "$.embed_text",
        "FLAT",
        {
            "TYPE": "FLOAT32",
            "DIM": VECTOR_DIMENSION,
            "DISTANCE_METRIC": "COSINE",
        },
        as_name="vector",
    ),
)
definition = IndexDefinition(prefix=["automobiles:"], index_type=IndexType.JSON)
res = client.ft("idx:automobiles_vss").create_index(fields=schema, definition=definition)

In [124]:
res

b'OK'

In [125]:
info = client.ft("idx:automobiles_vss").info()
num_docs = info["num_docs"]
indexing_failures = info["hash_indexing_failures"]

print(f"{num_docs} documents indexed with {indexing_failures} failures")

450 documents indexed with 0 failures
